# Gold: Earthquake Daily Summary (v1)

Reads the `earthquake_data.earthquakes` bronze table (see
`data-generators/bronze_ingest_usgs_earthquakes.ipynb`) and aggregates it
by event date: total quakes, average/max magnitude, and count of
magnitude-4.0+ events. Persists the result as a managed Spark table for
demo dashboards/queries.

Re-runnable: `mode("overwrite")`, so re-running just refreshes the table.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "earthquake_data"
SOURCE_TABLE = "earthquakes"
TARGET_TABLE = "gold_earthquake_daily_summary_v1"

## Load bronze table

In [ ]:
df = spark.table(f"{DATABASE_NAME}.{SOURCE_TABLE}")

## Transform: daily earthquake summary

In [ ]:
summary_df = (
    df
    .withColumn("event_date", F.to_date("event_time"))
    .groupBy("event_date")
    .agg(
        F.count("*").alias("total_quakes"),
        F.round(F.avg("magnitude"), 2).alias("avg_magnitude"),
        F.round(F.max("magnitude"), 2).alias("max_magnitude"),
        F.sum(F.when(F.col("magnitude") >= 4.0, 1).otherwise(0)).alias("quakes_mag4_plus"),
    )
    .orderBy("event_date")
)

## Persist as a managed Spark table

In [ ]:
summary_df.write \
    .mode("overwrite") \
    .saveAsTable(f"{DATABASE_NAME}.{TARGET_TABLE}")

## Verify

In [ ]:
spark.sql(f"SELECT COUNT(*) AS total_days FROM {DATABASE_NAME}.{TARGET_TABLE}").show()
spark.sql(f"SELECT * FROM {DATABASE_NAME}.{TARGET_TABLE} ORDER BY event_date DESC LIMIT 10").show()